# CivicPulse AI Complaint Classifier

This notebook documents the machine learning workflow used in CivicPulse for classifying citizen complaints into civic categories.

In [29]:

from sklearn.metrics.pairwise import cosine_similarity

DATASET LOADING

In [30]:
import pandas as pd
df = pd.read_csv("../data/complaints_dataset.csv")
df.head()

,complaint_text,category
0,Garbage not collected in Bahour,Sanitation
1,Stray dogs gathering near garbage dump in Vill...,Other
2,Road surface near Ariyankuppam is damaged and ...,Roads
3,Road damaged in Mudaliarpet,Roads
4,Power outage in Muthialpet,Electrical


In [31]:
df.shape

(510, 2)

In [32]:
df["category"].value_counts()

category
Other         107
Sanitation    104
Electrical    101
Roads          99
Water          99
Name: count, dtype: int64

In [33]:
X = df["complaint_text"]
y = df["category"]

SPLIT DATASET

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 331
Testing Samples: 179


## Model Pipeline

The model pipeline has two main steps:

- TF-IDF Vectorizer  
   Converts complaint text into numerical features.

- Logistic Regression  
   Classifies the complaint into one of the predefined categories.

## Train model

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [36]:
model = Pipeline([
    ("tfidf",TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.80)
    ),
    ("classifier",LogisticRegression(
            max_iter=1000,
            C=0.3)
    )
])

model.fit(X_train, y_train)

print("Model Training Complete")

Model Training Complete


In [37]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 98.32 %


HERE THE DATASET CONTAINS WELL DEFINED COMPLAINTS CATEGORIES , AND TF-IDF COMBINED WITH LOGISTIC REGRESSION BECOMES HIGHLY EFFECTIVE AND THUS THE ACCURACY IS 98.32% (NOT A CASE OF OVERFITTING).

## Testing

In [38]:
model.predict([
    "Streetlight not working near bus stand"
])

array(['Electrical'], dtype=object)

In [39]:
model.predict([
    "Garbage not collected for a week"
])

array(['Sanitation'], dtype=object)

In [40]:
import joblib

joblib.dump(
    model,
    "../models/complaint_classifier.pkl"
)

['../models/complaint_classifier.pkl']

## Urgency Detection

Urgency is detected using rule-based keyword matching.

In [41]:
def predict_urgency(text):
    text = text.lower()

    high_keywords = ["hospital","school","fire","overflow","accident","danger","electrical",
                     "transformer","leakage","emergency","sewage"]

    medium_keywords = ["road","garbage","streetlight","water","pothole","drainage"]

    if any(word in text for word in high_keywords):
        return "High"

    if any(word in text for word in medium_keywords):
        return "Medium"

    return "Low"

## Department Routing

After predicting the category, the complaint is mapped to the responsible department.

In [42]:
def predict_department(category):
    mapping = {
        "Water": "Public Works Department",
        "Roads": "Public Works Department",
        "Sanitation": "Municipality Sanitation Department",
        "Electrical": "Electricity Department",
        "Other": "Municipal Administration"
    }

    return mapping.get(category, "Municipal Administration")

## Full Complaint Classification

This combines category prediction, urgency detection, and department routing.

In [43]:
def classify_complaint(text):
    category = model.predict([text])[0]
    urgency = predict_urgency(text)
    department = predict_department(category)

    return {
        "category": category,
        "urgency": urgency,
        "department": department
    }

## Complaint Testing



In [44]:
test_cases = [
    "Sewage overflow near government school in Lawspet",
    "Streetlight not working near bus stand for a week",
    "Stray dogs nuisance in residential area",
    "Garbage not collected for 5 days near market",
    "Transformer sparking near residential area, danger",
    "Nuisance caused by open concerts in white town"
]

for text in test_cases:
    print("\nComplaint:", text)
    print("Prediction:", classify_complaint(text))


Complaint: Sewage overflow near government school in Lawspet
Prediction: {'category': 'Sanitation', 'urgency': 'High', 'department': 'Municipality Sanitation Department'}

Complaint: Streetlight not working near bus stand for a week
Prediction: {'category': 'Electrical', 'urgency': 'Medium', 'department': 'Electricity Department'}

Complaint: Stray dogs nuisance in residential area
Prediction: {'category': 'Other', 'urgency': 'Low', 'department': 'Municipal Administration'}

Complaint: Garbage not collected for 5 days near market
Prediction: {'category': 'Sanitation', 'urgency': 'Medium', 'department': 'Municipality Sanitation Department'}

Complaint: Transformer sparking near residential area, danger
Prediction: {'category': 'Electrical', 'urgency': 'High', 'department': 'Electricity Department'}

Complaint: Nuisance caused by open concerts in white town
Prediction: {'category': 'Sanitation', 'urgency': 'Low', 'department': 'Municipality Sanitation Department'}


## Duplicate Complaint Detection

In [45]:
class FakeComplaint:
    def __init__(self, title, description):
        self.title = title
        self.description = description


def check_duplicate(new_text, existing_complaints, threshold=0.75):
    if not existing_complaints:
        return None

    texts = [
        f"{complaint.title} {complaint.description}"
        for complaint in existing_complaints
    ]

    texts.append(new_text)

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2)
    )

    vectors = vectorizer.fit_transform(texts)

    similarities = cosine_similarity(
        vectors[-1],
        vectors[:-1]
    )[0]

    max_score = similarities.max()

    if max_score >= threshold:
        best_match_index = similarities.argmax()
        return existing_complaints[best_match_index]

    return None

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

existing_complaints = [
    FakeComplaint(
        "Water leakage",
        "Water leakage near Lawspet school"
    ),
    FakeComplaint(
        "Streetlight issue",
        "Streetlight not working near Muthialpet market"
    )
]

new_complaint = "Water pipe leakage near Lawspet school"

duplicate = check_duplicate(new_complaint, existing_complaints)

if duplicate:
    print("Duplicate warning: Similar complaint exists")
    print("Matched Complaint:", duplicate.title)
else:
    print("No duplicate found")

No duplicate found


## Model Summary

- Dataset: Puducherry-style civic complaints
- Categories: Water, Roads, Sanitation, Electrical, Other
- ML Pipeline: TF-IDF Vectorizer + Logistic Regression
- Output: Category, urgency, department
- Extra Feature: Duplicate complaint warning